In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# =========================
# Load datasets
# =========================
day_df = pd.read_csv("day.csv")
hour_df = pd.read_csv("hour.csv")

# =========================
# Filter selected months
# =========================
selected_months = [3, 6, 9, 12]
day_df = day_df[day_df['mnth'].isin(selected_months)]
hour_df = hour_df[hour_df['mnth'].isin(selected_months)]

# =========================
# ADD WEEKDAY & WEEKEND FEATURES
# =========================
day_df['dteday'] = pd.to_datetime(day_df['dteday'])
hour_df['dteday'] = pd.to_datetime(hour_df['dteday'])

day_df['weekday_num'] = day_df['dteday'].dt.weekday
day_df['is_weekend'] = day_df['weekday_num'].isin([5, 6]).astype(int)

hour_df['weekday_num'] = hour_df['dteday'].dt.weekday
hour_df['is_weekend'] = hour_df['weekday_num'].isin([5, 6]).astype(int)

# =========================
# DAY DATASET → DAILY BIKE RENTAL MODEL
# =========================
day_df = day_df.drop(['instant', 'casual', 'registered'], axis=1)

X_day = day_df.drop(['cnt', 'dteday'], axis=1)
y_day = day_df['cnt']

X_day_train, X_day_test, y_day_train, y_day_test = train_test_split(
    X_day, y_day, test_size=0.2, random_state=42
)

day_model = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)

day_model.fit(X_day_train, y_day_train)
y_day_pred = day_model.predict(X_day_test)

# =========================
# HOUR DATASET → HOURLY BIKE RENTAL MODEL
# =========================
hour_df = hour_df[hour_df['hr'].isin([7, 8, 9, 17, 18, 19])]
hour_df = hour_df.drop(['instant', 'casual', 'registered'], axis=1)

X_hour = hour_df.drop(['cnt', 'dteday'], axis=1)
y_hour = hour_df['cnt']

X_hour_train, X_hour_test, y_hour_train, y_hour_test = train_test_split(
    X_hour, y_hour, test_size=0.2, random_state=42
)

hour_model = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)

hour_model.fit(X_hour_train, y_hour_train)
y_hour_pred = hour_model.predict(X_hour_test)

# =========================
# COMBINED EVALUATION
# ⚠️ Scale difference still applies (expected)
# =========================
y_true_combined = np.concatenate([y_day_test.values, y_hour_test.values])
y_pred_combined = np.concatenate([y_day_pred, y_hour_pred])

mae = mean_absolute_error(y_true_combined, y_pred_combined)
rmse = np.sqrt(mean_squared_error(y_true_combined, y_pred_combined))
r2 = r2_score(y_true_combined, y_pred_combined)

print("Combined MAE:", mae)
print("Combined RMSE:", rmse)
print("Combined R2:", r2)

# =========================
# SAVE BOTH MODELS
# =========================
joblib.dump(day_model, "bike_demand_day_model.pkl")
joblib.dump(hour_model, "bike_demand_hour_model.pkl")

print("Models saved separately:")
print("- bike_demand_day_model.pkl")
print("- bike_demand_hour_model.pkl")


Combined MAE: 109.31005859375
Combined RMSE: 304.507029886011
Combined R2: 0.965969443321228
Models saved separately:
- bike_demand_day_model.pkl
- bike_demand_hour_model.pkl
